# 02 — The inputs, as known the day before each CPI release

**Reads:** `data/processed/monthly.csv`.

**Writes:** `data/processed/features.csv`; `results/tables/design_checks_lead.csv`,
`naive_benchmarks_before_2021.csv`, `timing_check.json`.

**The nowcast day.** Each nowcast is made on the day before the month's CPI release. That is roughly when
the Reuters poll is collected, so the models and the poll work from the same information. An input
qualifies only if it was published before that day.

**Target:** the first print of core CPI's monthly change, in percent (rounded to 0.1, as published).

**Inputs** (one row per reference month *t*):

| Column | What it is | Public before the release because |
|---|---|---|
| `core_last` | the most recent first print (normally month *t*−1) | it was released about a month earlier |
| `core_avg3` | average of the 3 most recent first prints | same |
| `core_avg12` | average of the 12 most recent first prints | same |
| `manheim_0`, `_1`, `_2` | % change in Manheim's wholesale used-car index in months *t*, *t*−1, *t*−2 | Manheim's full-month figure comes out on the 5th business day of the next month, before CPI (checked below) |
| `jet_0`, `_1` | % change in the monthly average jet fuel price in months *t*, *t*−1 | daily prices, so the whole month is known when it ends |

The core averages use the most recent prints that exist, so for December 2025 and January 2026 they skip
the two months that were never published.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
TAB = ROOT / "results" / "tables"
FIG = ROOT / "results" / "figures"
for d in (PROC, TAB, FIG):
    d.mkdir(parents=True, exist_ok=True)

from pandas.tseries.holiday import USFederalHolidayCalendar
from pandas.tseries.offsets import CustomBusinessDay

TEST_START = pd.Period("2021-01", freq="M")

monthly = pd.read_csv(PROC / "monthly.csv", parse_dates=["release_date"])
monthly["period"] = pd.PeriodIndex(monthly["period"], freq="M")
monthly = monthly.set_index("period")
y = monthly["core_first"]
prints = y.dropna()                    # months that have a first print, in calendar order

## One row per month: target, poll and inputs

In [2]:
rows = []
for t in monthly.index:
    if pd.isna(y[t]):
        continue                                   # no first print for this month: not a target
    before = prints.loc[: t - 1]                   # prints of earlier months, all released before t's
    if len(before) < 12:
        continue                                   # not enough history for the 12-print average
    last12 = before.iloc[-12:]
    rows.append({
        "period": t,
        "release_date": monthly.at[t, "release_date"],
        "y": y[t],
        "poll": monthly.at[t, "poll"],
        "core_last": last12.iloc[-1],
        "core_avg3": last12.iloc[-3:].mean(),
        "core_avg12": last12.mean(),
        "last_print_month": last12.index[-1],
        "manheim_0": monthly.at[t, "manheim_mom"],
        "manheim_1": monthly.at[t - 1, "manheim_mom"],
        "manheim_2": monthly.at[t - 2, "manheim_mom"],
        "jet_0": monthly.at[t, "jet_mom"],
        "jet_1": monthly.at[t - 1, "jet_mom"],
    })
feat = pd.DataFrame(rows).set_index("period")
feat["nowcast_date"] = feat["release_date"] - pd.Timedelta(days=1)

INPUTS = ["core_last", "core_avg3", "core_avg12", "manheim_0", "manheim_1", "manheim_2", "jet_0", "jet_1"]
assert feat[INPUTS].notna().all().all()
print(len(feat), "months,", feat.index.min(), "to", feat.index.max())
print("before the test:", int((feat.index < TEST_START).sum()), "  test:", int((feat.index >= TEST_START).sum()))
feat.loc[[pd.Period(p, freq="M") for p in ("2025-09", "2025-12", "2026-01")], ["y", "core_last", "last_print_month", "core_avg3"]]

257 months, 2005-04 to 2026-10
before the test: 189   test: 68


,y,core_last,last_print_month,core_avg3
period,,,,
2025-09,0.2,0.3,2025-08,0.266667
2025-12,0.2,0.2,2025-09,0.266667
2026-01,0.3,0.2,2025-12,0.233333


## Timing check: was every input public on the nowcast day?

Core prints: each one used was released before the nowcast date. Manheim: the full-month index comes out
on the 5th business day of the following month; the check uses the U.S. federal holiday calendar.

In [3]:
release = monthly["release_date"]
used_print_release = release.reindex(feat["last_print_month"]).to_numpy()
assert (used_print_release < feat["nowcast_date"].to_numpy()).all()

bday = CustomBusinessDay(calendar=USFederalHolidayCalendar())


def fifth_business_day(p):
    start = p.to_timestamp(how="start")
    return pd.date_range(start, start + pd.Timedelta(days=15), freq=bday)[4]


feat["manheim_release_est"] = [fifth_business_day(t + 1) for t in feat.index]
late = feat.index[feat["manheim_release_est"] > feat["nowcast_date"]]
timing = {
    "core prints used: all released before the nowcast date": True,
    "months where Manheim's estimated release falls after the nowcast date": [str(p) for p in late],
    "smallest gap, Manheim release to nowcast date (days)": int((feat["nowcast_date"] - feat["manheim_release_est"]).dt.days.min()),
}
(TAB / "timing_check.json").write_text(json.dumps(timing, indent=2))
timing

{'core prints used: all released before the nowcast date': True,
 "months where Manheim's estimated release falls after the nowcast date": [],
 'smallest gap, Manheim release to nowcast date (days)': 1}

## Design checks, on the months before the test only

These two checks were used to fix the inputs and the naive benchmark. They use months up to December 2020
only, so the test months played no part in either choice.

1. **How far ahead do used-car prices move?** Correlation of this month's change with Manheim's change *k*
   months earlier, for the CPI used-car index and for core CPI itself.
2. **Which naive rule is hardest to beat?** Error of three rules that need no model, and of the poll.

In [4]:
PRE = slice("2005-04", "2020-12")


def corr_pre(a, b):
    both = pd.concat([a, b], axis=1).loc[PRE].dropna()
    return float(both.corr().iloc[0, 1])


lead = pd.DataFrame([{
    "months earlier (k)": k,
    "CPI used cars vs Manheim": corr_pre(monthly["used_cars_mom"], monthly["manheim_mom"].shift(k)),
    "core CPI first print vs Manheim": corr_pre(y, monthly["manheim_mom"].shift(k)),
    "core CPI first print vs jet fuel": corr_pre(y, monthly["jet_mom"].shift(k)),
} for k in range(5)]).set_index("months earlier (k)")
lead.round(3).to_csv(TAB / "design_checks_lead.csv")
lead.round(2)

,CPI used cars vs Manheim,core CPI first print vs Manheim,core CPI first print vs jet fuel
months earlier (k),,,
0,-0.00,0.25,0.24
1,0.14,0.30,0.30
2,0.43,0.15,0.26
3,0.42,-0.10,-0.03
4,0.22,-0.15,-0.14


Manheim leads the CPI's own used-car index by about two to three months, but its link to **core** CPI is
strongest in the same month and one month later (and fades by month two). Jet fuel's link to core lasts about
two months. The inputs therefore use Manheim at 0–2 months and jet fuel at 0–1 months.

In [5]:
pre = feat.loc[: TEST_START - 1]
naive = {
    "Last month's print (persistence)": pre["core_last"],
    "Average of the last 3 prints": pre["core_avg3"],
    "Average of the last 12 prints": pre["core_avg12"],
    "Reuters poll (median)": pre["poll"],
}
rows = []
for name, f in naive.items():
    e = (pre["y"] - f).dropna()
    rows.append({"rule": name, "RMSE": float(np.sqrt((e ** 2).mean())), "MAE": float(e.abs().mean()), "months": len(e)})
naive_tab = pd.DataFrame(rows).set_index("rule")
naive_tab.round(4).to_csv(TAB / "naive_benchmarks_before_2021.csv")
naive_tab.round(3)

,RMSE,MAE,months
rule,,,
Last month's print (persistence),0.119,0.084,189
Average of the last 3 prints,0.125,0.089,189
Average of the last 12 prints,0.110,0.081,189
Reuters poll (median),0.094,0.068,184


Before 2021, last month's print was the *worst* naive rule: core's monthly prints are noisy (and rounded),
so the 12-print average beat it. Both are kept as benchmarks: persistence because it is the natural
"no change" rule, the 12-print average because it was the best rule without a model before the test.

In [6]:
out = feat.copy()
out.index = out.index.astype(str)
out["last_print_month"] = out["last_print_month"].astype(str)
out.drop(columns=["manheim_release_est"]).to_csv(PROC / "features.csv", float_format="%.6f")
out.tail(3)

,release_date,y,poll,core_last,core_avg3,core_avg12,last_print_month,manheim_0,manheim_1,manheim_2,jet_0,jet_1,nowcast_date,manheim_release_est
period,,,,,,,,,,,,,,
2026-08,2026-09-11,0.3,0.2,0.2,0.133333,0.225000,2026-07,-0.857143,-1.362142,0.141110,9.432853,11.867193,2026-09-10,2026-09-08
2026-09,2026-10-14,0.3,0.2,0.3,0.166667,0.233333,2026-08,-1.104707,-0.857143,-1.362142,17.830290,9.432853,2026-10-13,2026-10-07
2026-10,2026-11-10,0.3,0.2,0.3,0.266667,0.233333,2026-09,0.437105,-1.104707,-0.857143,2.164995,17.830290,2026-11-09,2026-11-06
